# Road Damage Detection — YOLO Train, Compare, Deploy

Before running: **Runtime > Change runtime type > T4 GPU**. Run cells top to bottom.

In [ ]:
!pip install -q -U ultralytics roboflow gradio

In [ ]:
import os
import glob
import time
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from roboflow import Roboflow

## 1. Dataset

Go to your dataset link → Download Dataset → format **YOLOv8** → "show download code", then paste that snippet in place of the block below (it already contains your API key, workspace, project and version).

In [ ]:
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("yolov8")

DATA_YAML = os.path.join(dataset.location, "data.yaml")

In [ ]:
for split_name in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(dataset.location, split_name, "images", "*"))
    print(split_name, len(imgs))

## 2. Config

If the dataset is large and training looks slow after the first epoch, lower `EPOCHS`, `TUNE_EPOCHS`, or `TRAIN_FRACTION` before continuing.

In [ ]:
MODEL_LIST = ["yolov8n.pt", "yolo11n.pt", "yolo26n.pt"]
EPOCHS = 25
TUNE_EPOCHS = 5
IMG_SIZE = 640
BATCH = 16
TRAIN_FRACTION = 1.0
EVAL_SPLIT = "valid"
TUNE_CONFIGS = [
    {"lr0": 0.01, "optimizer": "SGD"},
    {"lr0": 0.001, "optimizer": "AdamW"},
]

## 3. Train + light hyperparameter search + evaluate

For each model: 2 short tuning trials pick the better of two hyperparameter sets, then a full training run uses that config. Given the time budget this is a targeted search rather than an exhaustive one.

In [ ]:
def tune_and_train(model_name):
    best_cfg = TUNE_CONFIGS[0]
    best_score = -1
    for cfg in TUNE_CONFIGS:
        trial = YOLO(model_name)
        trial.train(
            data=DATA_YAML,
            epochs=TUNE_EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            fraction=TRAIN_FRACTION,
            lr0=cfg["lr0"],
            optimizer=cfg["optimizer"],
            project="tune_runs",
            name=f"{model_name.replace('.pt', '')}_{cfg['optimizer']}",
            plots=False,
            verbose=False,
            exist_ok=True,
        )
        trial_metrics = trial.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False)
        score = trial_metrics.box.map
        if score > best_score:
            best_score = score
            best_cfg = cfg

    final_model = YOLO(model_name)
    final_model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        lr0=best_cfg["lr0"],
        optimizer=best_cfg["optimizer"],
        project="final_runs",
        name=model_name.replace(".pt", ""),
        plots=True,
        exist_ok=True,
    )
    weights_path = os.path.join("final_runs", model_name.replace(".pt", ""), "weights", "best.pt")
    return weights_path, best_cfg

In [ ]:
def measure_fps(weights_path, n_samples=30):
    eval_model = YOLO(weights_path)
    sample_imgs = []
    for split_name in ["valid", "test", "train"]:
        found = glob.glob(os.path.join(dataset.location, split_name, "images", "*"))
        if found:
            sample_imgs = found[:n_samples]
            break
    eval_model.predict(sample_imgs[0], verbose=False)
    start = time.time()
    for img_path in sample_imgs:
        eval_model.predict(img_path, verbose=False)
    elapsed = time.time() - start
    return len(sample_imgs) / elapsed

In [ ]:
comparison_rows = []

for model_name in MODEL_LIST:
    weights_path, cfg_used = tune_and_train(model_name)
    eval_model = YOLO(weights_path)
    val_metrics = eval_model.val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=IMG_SIZE, verbose=False)
    fps = measure_fps(weights_path)
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)

    comparison_rows.append({
        "model": model_name,
        "lr0": cfg_used["lr0"],
        "optimizer": cfg_used["optimizer"],
        "precision": val_metrics.box.mp,
        "recall": val_metrics.box.mr,
        "mAP50": val_metrics.box.map50,
        "mAP50_95": val_metrics.box.map,
        "fps": fps,
        "size_mb": size_mb,
        "weights_path": weights_path,
    })
    print(model_name, "done")

## 4. Comparison table and plots

In [ ]:
df = pd.DataFrame(comparison_rows)
df["efficiency_score"] = df["fps"] / df["size_mb"]
df = df.sort_values("mAP50_95", ascending=False).reset_index(drop=True)
df.to_csv("model_comparison.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].bar(df["model"], df["mAP50_95"])
axes[0].set_title("mAP50-95")
axes[1].bar(df["model"], df["fps"])
axes[1].set_title("Inference FPS")
axes[2].bar(df["model"], df["size_mb"])
axes[2].set_title("Model Size (MB)")
axes[3].bar(df["model"], df["efficiency_score"])
axes[3].set_title("FPS per MB (efficiency)")
plt.tight_layout()
plt.savefig("model_comparison.png")
plt.show()

## 5. Pick the best model

In [ ]:
best_row = df.iloc[0]
shutil.copy(best_row["weights_path"], "best_model.pt")
print("Best model:", best_row["model"])
print(best_row)

## 6. Deployment app (Gradio)

Runs inline and gives a public share link.

In [ ]:
import gradio as gr
from PIL import Image

deploy_model = YOLO("best_model.pt")

def detect_damage(image, conf):
    results = deploy_model.predict(image, conf=conf, verbose=False)
    annotated = results[0].plot()[..., ::-1]
    names = results[0].names
    counts = {}
    for c in results[0].boxes.cls.tolist():
        label = names[int(c)]
        counts[label] = counts.get(label, 0) + 1
    summary = ", ".join(f"{k}: {v}" for k, v in counts.items()) if counts else "No damage detected"
    return Image.fromarray(annotated), summary

demo = gr.Interface(
    fn=detect_damage,
    inputs=[gr.Image(type="pil", label="Upload Road Image"), gr.Slider(0.1, 0.9, value=0.25, label="Confidence Threshold")],
    outputs=[gr.Image(type="pil", label="Detected Damage"), gr.Textbox(label="Detected Classes")],
    title="Road Damage Detection",
)
demo.launch(share=True)